Using EfficientNetB0 by Prince kumar

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

IMAGE_SIZE = 224
BATCH_SIZE = 32

# --------------------- Data Augmentation ---------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    "data",
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "data",
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

# --------------------- EfficientNetB0 ---------------------
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=8
)

# --------------------- Fine-Tune ---------------------
base_model.trainable = True

for layer in base_model.layers[:200]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)


Found 8023 images belonging to 2 classes.
Found 2005 images belonging to 2 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8
251/251 ━━━━━━━━━━━━━━━━━━━━ 185s 634ms/step - accuracy: 0.7910 - loss: 0.5231 - val_accuracy: 0.7985 - val_loss: 0.5033
Epoch 2/8
251/251 ━━━━━━━━━━━━━━━━━━━━ 127s 508ms/step - accuracy: 0.8024 - loss: 0.5055 - val_accuracy: 0.7985 - val_loss: 0.5025
Epoch 3/8
251/251 ━━━━━━━━━━━━━━━━━━━━ 127s 506ms/step - accuracy: 0.7824 - loss: 0.5328 - val_accuracy: 0.7985 - val_loss: 0.5102
Epoch 4/8
251/251 ━━━━━━━━━━━━━━━━━━━━ 125s 500ms/step - accuracy: 0.7960 - loss: 0.5116 - val_accuracy: 0.7985 - val_loss: 0.5060
Epoch 5/8
251/251 ━━━━━━━━━━━━━━━━━━━━ 127s 505ms/step - accuracy: 0.7972 - loss: 0.5124 - val_accuracy: 0.7985 - val_loss: 0.5131
Epoch 6/8
251/251 ━━━━━━━━━━━━━━━━━━━━ 126s 503ms/step - accuracy: 0.7970 - loss: 0.5116 - val_accuracy: 0.7985 - val_loss: 0.5027
Epoch 7/8
251/251 ━━━━━━━━━━━━━━━━━━━━ 127s 505ms/step - accuracy: 0.7979 - los

# New Section